# 02 - RAG System Setup

In this notebook, we'll:
1. Load the embeddings we created
2. Build a FAISS vector database
3. Test similarity search
4. Augment items with RAG features
5. Analyze the impact of similar products

In [ ]:
# Imports
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.items import Item
from src.rag import RAGSystem, build_rag_system
from src.embeddings import EmbeddingGenerator

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Step 1: Load Data and Embeddings

In [ ]:
# Load items
print("Loading items...")
train, val, test = Item.from_hub("ed-donner/items_lite")

# Enhance items
print("\nEnhancing items...")
for item in tqdm(train + val + test):
    item.enhance()

print(f"\nLoaded {len(train):,} train, {len(val):,} val, {len(test):,} test items")

In [ ]:
# Load embeddings
print("Loading embeddings...")
train_embeddings = np.load('../data/embeddings/train_embeddings.npy')
val_embeddings = np.load('../data/embeddings/val_embeddings.npy')
test_embeddings = np.load('../data/embeddings/test_embeddings.npy')

print(f"Train embeddings: {train_embeddings.shape}")
print(f"Val embeddings: {val_embeddings.shape}")
print(f"Test embeddings: {test_embeddings.shape}")

## Step 2: Build RAG System

We'll use FAISS to create a fast vector database for similarity search

In [ ]:
# Build RAG system with training data
print("Building RAG system...\n")
rag_system = build_rag_system(
    embeddings=train_embeddings,
    items=train,
    index_type='flat'  # Use 'ivf' for faster approximate search on larger datasets
)

print("\n✅ RAG system ready!")

## Step 3: Test Similarity Search

In [ ]:
# Pick a test item
test_item = test[0]
test_embedding = test_embeddings[0]

print("Query Product:")
print(f"Title: {test_item.title}")
print(f"Category: {test_item.category}")
print(f"Price: ${test_item.price:.2f}")
print(f"\nSummary: {test_item.summary}")

In [ ]:
# Find similar products
similar_products = rag_system.search(test_embedding, k=10, exclude_self=False)

print("\n" + "="*80)
print("TOP 10 SIMILAR PRODUCTS")
print("="*80)

for i, prod in enumerate(similar_products, 1):
    print(f"\n{i}. {prod.title}")
    print(f"   Category: {prod.category}")
    print(f"   Price: ${prod.price:.2f}")
    print(f"   Similarity: {prod.similarity:.4f}")

## Step 4: Analyze Price Statistics from Similar Products

In [ ]:
# Get price statistics
stats = rag_system.get_price_statistics(similar_products)
weighted_price = rag_system.get_weighted_price(similar_products)

print("\nPrice Statistics from Similar Products:")
print(f"Mean: ${stats['mean']:.2f}")
print(f"Median: ${stats['median']:.2f}")
print(f"Std Dev: ${stats['std']:.2f}")
print(f"Min: ${stats['min']:.2f}")
print(f"Max: ${stats['max']:.2f}")
print(f"Weighted Average: ${weighted_price:.2f}")
print(f"\nActual Price: ${test_item.price:.2f}")
print(f"Error (Mean): ${abs(stats['mean'] - test_item.price):.2f}")
print(f"Error (Weighted): ${abs(weighted_price - test_item.price):.2f}")

In [ ]:
# Visualize similar product prices
prices = [p.price for p in similar_products]
similarities = [p.similarity for p in similar_products]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Price distribution
ax1.bar(range(len(prices)), prices, color='skyblue')
ax1.axhline(y=test_item.price, color='red', linestyle='--', label=f'Actual: ${test_item.price:.2f}')
ax1.axhline(y=stats['mean'], color='green', linestyle='--', label=f'Mean: ${stats["mean"]:.2f}')
ax1.set_xlabel('Similar Product Rank')
ax1.set_ylabel('Price ($)')
ax1.set_title('Prices of Similar Products')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Similarity scores
ax2.bar(range(len(similarities)), similarities, color='coral')
ax2.set_xlabel('Similar Product Rank')
ax2.set_ylabel('Similarity Score')
ax2.set_title('Similarity Scores')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Augment All Items with RAG Features

In [ ]:
# Augment training items (using leave-one-out)
print("Augmenting training items with RAG features...")
for idx, item in enumerate(tqdm(train)):
    rag_system.augment_item_with_rag(item, idx, k=10)

print("\n✅ Training items augmented!")

In [ ]:
# For validation and test, we search in the training set
print("Augmenting validation items...")
for idx, item in enumerate(tqdm(val)):
    query_emb = val_embeddings[idx]
    similar = rag_system.search(query_emb, k=10, exclude_self=False)
    
    item.similar_prices = [p.price for p in similar]
    stats = rag_system.get_price_statistics(similar)
    item.rag_mean_price = stats['mean']
    item.rag_median_price = stats['median']
    item.rag_std_price = stats['std']
    item.rag_weighted_price = rag_system.get_weighted_price(similar)

print("\nAugmenting test items...")
for idx, item in enumerate(tqdm(test)):
    query_emb = test_embeddings[idx]
    similar = rag_system.search(query_emb, k=10, exclude_self=False)
    
    item.similar_prices = [p.price for p in similar]
    stats = rag_system.get_price_statistics(similar)
    item.rag_mean_price = stats['mean']
    item.rag_median_price = stats['median']
    item.rag_std_price = stats['std']
    item.rag_weighted_price = rag_system.get_weighted_price(similar)

print("\n✅ All items augmented with RAG features!")

## Step 6: Analyze RAG Feature Quality

In [ ]:
# Check RAG features on test set
sample = test[0]
print("Sample item with RAG features:")
print(f"Title: {sample.title}")
print(f"Actual Price: ${sample.price:.2f}")
print(f"\nRAG Features:")
print(f"  Mean of similar: ${sample.rag_mean_price:.2f}")
print(f"  Median of similar: ${sample.rag_median_price:.2f}")
print(f"  Std of similar: ${sample.rag_std_price:.2f}")
print(f"  Weighted average: ${sample.rag_weighted_price:.2f}")
print(f"\nSimilar prices: {[f'${p:.2f}' for p in sample.similar_prices[:5]]}")

In [ ]:
# Evaluate RAG-based prediction (simple baseline)
from src.evaluator import compute_metrics

# Use RAG mean as prediction
rag_predictions = [item.rag_mean_price for item in test]
actual_prices = [item.price for item in test]

metrics = compute_metrics(rag_predictions, actual_prices)

print("\n" + "="*60)
print("RAG-ONLY BASELINE (using mean of similar products)")
print("="*60)
print(f"MAE: ${metrics['mae']:.2f}")
print(f"RMSE: ${metrics['rmse']:.2f}")
print(f"MAPE: {metrics['mape']:.2f}%")
print(f"R²: {metrics['r2']:.4f}")
print(f"Within 10%: {metrics['within_10_pct']:.1f}%")
print(f"Within 20%: {metrics['within_20_pct']:.1f}%")
print("="*60)

In [ ]:
# Visualize RAG prediction quality
errors = [abs(pred - actual) for pred, actual in zip(rag_predictions, actual_prices)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Scatter plot: predicted vs actual
ax1.scatter(actual_prices, rag_predictions, alpha=0.5, s=20)
ax1.plot([0, max(actual_prices)], [0, max(actual_prices)], 'r--', label='Perfect prediction')
ax1.set_xlabel('Actual Price ($)')
ax1.set_ylabel('RAG Predicted Price ($)')
ax1.set_title('RAG Predictions vs Actual Prices')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Error distribution
ax2.hist(errors, bins=50, color='coral', edgecolor='black')
ax2.axvline(x=metrics['mae'], color='red', linestyle='--', label=f'MAE: ${metrics["mae"]:.2f}')
ax2.set_xlabel('Absolute Error ($)')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Prediction Errors')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Save RAG-Enhanced Data

In [ ]:
# Save RAG features
import pickle

# Save the RAG system
with open('../data/rag_system.pkl', 'wb') as f:
    pickle.dump(rag_system, f)

# Save augmented items
with open('../data/train_items_rag.pkl', 'wb') as f:
    pickle.dump(train, f)

with open('../data/val_items_rag.pkl', 'wb') as f:
    pickle.dump(val, f)

with open('../data/test_items_rag.pkl', 'wb') as f:
    pickle.dump(test, f)

print("✅ RAG system and augmented items saved!")
print("\nFiles created:")
print("- data/rag_system.pkl")
print("- data/train_items_rag.pkl")
print("- data/val_items_rag.pkl")
print("- data/test_items_rag.pkl")

## Summary

✅ Built FAISS vector database  
✅ Tested similarity search  
✅ Augmented all items with RAG features  
✅ Evaluated RAG-only baseline  
✅ Saved RAG system and data  

**Key Finding:** RAG-only (using mean of similar products) achieves decent accuracy!  
This will be a powerful feature for our ML models.

**Next:** Notebook 03 - Train XGBoost and Neural Network models